### Mount Drive, initialise Spark, and set the verified Task 1 Parquet path.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json, time, math
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler, StandardScaler

spark = (
    SparkSession.builder
    .appName("7006SCN_Task2_WESAD_Data_Engineering")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

OUTPUT = Path("/content/drive/MyDrive/wesad_spark_parquet_fresh_run")
MANIFEST_PATH = OUTPUT / "_verification_manifest.json"
EXPECTED_SUBJECTS = [
    "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10", "S11",
    "S13", "S14", "S15", "S16", "S17"
]
FEATURES = ["ACC_X", "ACC_Y", "ACC_Z", "ECG", "EMG", "EDA", "Temp", "Resp"]
REQUIRED_COLUMNS = FEATURES + ["label", "subject_id", "sample_index"]

assert OUTPUT.exists(), f"Task 1 output folder not found: {OUTPUT}"
assert MANIFEST_PATH.exists(), f"Task 1 verification manifest not found: {MANIFEST_PATH}"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
assert manifest.get("status") == "verified", "Task 1 manifest is not marked verified."
assert manifest.get("participant_count") == 15, "Task 1 manifest does not record 15 participants."
print("Spark version:", spark.version)
print("Verified source folder:", OUTPUT)
print("Manifest status:", manifest["status"])

Mounted at /content/drive
Spark version: 4.0.4
Verified source folder: /content/drive/MyDrive/wesad_spark_parquet_fresh_run
Manifest status: verified


### Load the 15 saved participant Parquet folders and verify schema, rows, and partition count.


In [ ]:
participant_paths = [OUTPUT / s for s in EXPECTED_SUBJECTS]
missing_paths = [str(p) for p in participant_paths if not p.exists() or not list(p.rglob("*.parquet"))]
if missing_paths:
    raise FileNotFoundError(f"Missing participant Parquet outputs: {missing_paths}")

df_raw = spark.read.parquet(*[str(p) for p in participant_paths])
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Unexpected Task 1 schema; missing columns: {missing_cols}")

actual_subjects = sorted(r["subject_id"] for r in df_raw.select("subject_id").distinct().collect())
if actual_subjects != sorted(EXPECTED_SUBJECTS):
    raise ValueError(f"Participant IDs mismatch: {actual_subjects}")

raw_rows = df_raw.count()
print("Participant folders loaded:", len(participant_paths))
print("Distinct participant IDs:", len(actual_subjects), actual_subjects)
print("Rows loaded:", f"{raw_rows:,}")
print("Columns:", df_raw.columns)
print("Input partitions:", df_raw.rdd.getNumPartitions())
df_raw.printSchema()
df_raw.show(20, truncate=False)

Participant folders loaded: 15
Distinct participant IDs: 15 ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Rows loaded: 60,807,600
Columns: ['ACC_X', 'ACC_Y', 'ACC_Z', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp', 'label', 'subject_id', 'sample_index']
Input partitions: 16
root
 |-- ACC_X: double (nullable = true)
 |-- ACC_Y: double (nullable = true)
 |-- ACC_Z: double (nullable = true)
 |-- ECG: double (nullable = true)
 |-- EMG: double (nullable = true)
 |-- EDA: double (nullable = true)
 |-- Temp: double (nullable = true)
 |-- Resp: double (nullable = true)
 |-- label: integer (nullable = true)
 |-- subject_id: string (nullable = true)
 |-- sample_index: long (nullable = true)

+---------------------+--------------------+-------------------+-------------------+-------------------+------------------+------------------+----------------+-----+----------+------------+
|ACC_X                |ACC_Y               |ACC_Z              |ECG           

### Inspect missing, non-finite, label, and duplicate records before preprocessing.


In [ ]:
numeric_cols = FEATURES + ["label", "sample_index"]

missing_exprs = []
for c in numeric_cols:
    missing_exprs.append(
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.isnan(F.col(c).cast("double")) if c in FEATURES else F.lit(False)),
                1
            ).otherwise(0)
        ).alias(c)
    )
missing_before = df_raw.agg(*missing_exprs).first().asDict()
print("Missing/null/NaN counts before cleaning:")
for k, v in missing_before.items():
    print(f"{k}: {v:,}")

print("Label distribution before filtering:")
df_raw.groupBy("label").count().orderBy("label").show(20, truncate=False)

print("Duplicate sample keys (subject_id, sample_index):")
duplicate_keys = (
    df_raw.groupBy("subject_id", "sample_index")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate keys: {duplicate_keys:,}")

Missing/null/NaN counts before cleaning:
ACC_X: 0
ACC_Y: 0
ACC_Z: 0
ECG: 0
EMG: 0
EDA: 0
Temp: 0
Resp: 0
label: 0
sample_index: 0
Label distribution before filtering:
+-----+--------+
|label|count   |
+-----+--------+
|0    |27654897|
|1    |12327702|
|2    |6976201 |
|3    |3902501 |
|4    |8264199 |
|5    |552300  |
|6    |552998  |
|7    |576802  |
+-----+--------+

Duplicate sample keys (subject_id, sample_index):
Duplicate keys: 0


### Clean invalid numeric values and retain the four labelled WESAD states used for stress classification.


In [ ]:
# Convert NaN and +/- infinity in signal features to null so the imputer can handle them.
df_clean = df_raw
for c in FEATURES:
    x = F.col(c).cast("double")
    df_clean = df_clean.withColumn(
        c,
        F.when(
            x.isNull() | F.isnan(x) | (F.abs(x) == float("inf")),
            F.lit(None).cast("double")
        ).otherwise(x)
    )

# WESAD labels: 1=baseline, 2=stress, 3=amusement, 4=meditation.
# Exclude label 0 (undefined) and transitional labels from this binary task.
df_clean = (
    df_clean
    .withColumn("label_int", F.col("label").cast("int"))
    .filter(F.col("label_int").isin([1, 2, 3, 4]))
    .withColumn(
        "target_text",
        F.when(F.col("label_int") == 2, F.lit("stress")).otherwise(F.lit("non_stress"))
    )
    .select(*FEATURES, "label_int", "target_text", "subject_id", "sample_index")
)

rows_after_filter = df_clean.count()
print("Rows before preprocessing:", f"{raw_rows:,}")
print("Rows after removing undefined/transitional labels:", f"{rows_after_filter:,}")
print("Rows excluded:", f"{raw_rows - rows_after_filter:,}")
print("Binary target distribution:")
df_clean.groupBy("target_text").count().orderBy("target_text").show()

Rows before preprocessing: 60,807,600
Rows after removing undefined/transitional labels: 31,470,603
Rows excluded: 29,336,997
Binary target distribution:
+-----------+--------+
|target_text|   count|
+-----------+--------+
| non_stress|24494402|
|     stress| 6976201|
+-----------+--------+



### Split by participant to reduce subject leakage and report train/test partition counts.


In [ ]:
# Hold out complete participants so the same participant's samples do not occur in both sets.
TEST_SUBJECTS = ["S15", "S16", "S17"]
train_subjects = [s for s in EXPECTED_SUBJECTS if s not in TEST_SUBJECTS]

train_raw = df_clean.filter(~F.col("subject_id").isin(TEST_SUBJECTS))
test_raw = df_clean.filter(F.col("subject_id").isin(TEST_SUBJECTS))

train_rows = train_raw.count()
test_rows = test_raw.count()

if train_rows == 0 or test_rows == 0:
    raise RuntimeError("Train/test split produced an empty partition.")
if train_raw.select("subject_id").distinct().intersect(
    test_raw.select("subject_id").distinct()
).count() != 0:
    raise RuntimeError("Participant leakage detected between train and test sets.")

print("Training participants:", train_subjects)
print("Test participants:", TEST_SUBJECTS)
print(f"Training rows: {train_rows:,}")
print(f"Test rows: {test_rows:,}")
print("Training Spark partitions:", train_raw.rdd.getNumPartitions())
print("Test Spark partitions:", test_raw.rdd.getNumPartitions())
print("Train class distribution:")
train_raw.groupBy("target_text").count().show()
print("Test class distribution:")
test_raw.groupBy("target_text").count().show()

Training participants: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14']
Test participants: ['S15', 'S16', 'S17']
Training rows: 25,137,704
Test rows: 6,332,899
Training Spark partitions: 16
Test Spark partitions: 16
Train class distribution:
+-----------+--------+
|target_text|   count|
+-----------+--------+
|     stress| 5518800|
| non_stress|19618904|
+-----------+--------+

Test class distribution:
+-----------+-------+
|target_text|  count|
+-----------+-------+
|     stress|1457401|
| non_stress|4875498|
+-----------+-------+



### Build and fit a reusable PySpark preprocessing pipeline on training data only.


In [ ]:
# Median imputation is fitted on training data only.
imputer = Imputer(
    inputCols=FEATURES,
    outputCols=[f"{c}_imputed" for c in FEATURES],
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in FEATURES],
    outputCol="features_raw",
    handleInvalid="error"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=False,
    withStd=True
)

label_indexer = StringIndexer(
    inputCol="target_text",
    outputCol="label",
    handleInvalid="error",
    stringOrderType="alphabetAsc"
)

pipeline = Pipeline(stages=[imputer, assembler, scaler, label_indexer])

fit_started = time.time()
pipeline_model = pipeline.fit(train_raw)
fit_seconds = time.time() - fit_started

train_processed = pipeline_model.transform(train_raw).select(
    "features_scaled", "label", "target_text", "subject_id", "sample_index"
).withColumnRenamed("features_scaled", "features")

test_processed = pipeline_model.transform(test_raw).select(
    "features_scaled", "label", "target_text", "subject_id", "sample_index"
).withColumnRenamed("features_scaled", "features")

print(f"Pipeline fit time: {fit_seconds:.2f} seconds")
print("Pipeline stages:")
for i, stage in enumerate(pipeline_model.stages, start=1):
    print(i, type(stage).__name__)
print("Processed training rows:", f"{train_processed.count():,}")
print("Processed test rows:", f"{test_processed.count():,}")
train_processed.printSchema()
train_processed.show(5, truncate=False)

Pipeline fit time: 180.92 seconds
Pipeline stages:
1 ImputerModel
2 VectorAssembler
3 StandardScalerModel
4 StringIndexerModel
Processed training rows: 25,137,704
Processed test rows: 6,332,899
root
 |-- features: vector (nullable = true)
 |-- label: double (nullable = false)
 |-- target_text: string (nullable = false)
 |-- subject_id: string (nullable = true)
 |-- sample_index: long (nullable = true)

+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-----------+----------+------------+
|features                                                                                                                                                 |label|target_text|subject_id|sample_index|
+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-----------+----------+------------+


### Verify processed data quality and save the fitted pipeline and Task 2 outputs to Drive.


In [ ]:
from pyspark.sql import functions as F

for name, frame in [("train", train_processed), ("test", test_processed)]:
    null_features = frame.filter(F.col("features").isNull()).count()
    null_labels = frame.filter(F.col("label").isNull()).count()
    print(f"{name}: null feature vectors={null_features:,}; null labels={null_labels:,}")
    if null_features or null_labels:
        raise RuntimeError(f"Processed {name} data contains null features or labels.")

RESULTS = Path("/content/drive/MyDrive/wesad_task2_outputs")
RESULTS.mkdir(parents=True, exist_ok=True)

pipeline_path = RESULTS / "preprocessing_pipeline"
pipeline_model.write().overwrite().save(str(pipeline_path))

train_out = RESULTS / "train_processed"
test_out = RESULTS / "test_processed"
train_processed.write.mode("overwrite").parquet(str(train_out))
test_processed.write.mode("overwrite").parquet(str(test_out))

task2_manifest = {
    "source_parquet_folder": str(OUTPUT),
    "source_task1_manifest_status": manifest.get("status"),
    "source_participant_count": len(actual_subjects),
    "source_rows_loaded": int(raw_rows),
    "rows_after_label_filter": int(rows_after_filter),
    "training_rows": int(train_rows),
    "test_rows": int(test_rows),
    "training_participants": train_subjects,
    "test_participants": TEST_SUBJECTS,
    "input_partitions": int(df_raw.rdd.getNumPartitions()),
    "training_partitions": int(train_raw.rdd.getNumPartitions()),
    "test_partitions": int(test_raw.rdd.getNumPartitions()),
    "pipeline_stages": [type(stage).__name__ for stage in pipeline_model.stages],
    "pipeline_fit_seconds": round(fit_seconds, 2),
    "features": FEATURES,
    "target_definition": "label 2 = stress; labels 1, 3, 4 = non_stress; labels 0 and 5+ excluded",
    "status": "verified"
}
(RESULTS / "task2_manifest.json").write_text(json.dumps(task2_manifest, indent=2), encoding="utf-8")

print("Saved pipeline:", pipeline_path)
print("Saved train data:", train_out)
print("Saved test data:", test_out)
print("Saved manifest:", RESULTS / "task2_manifest.json")
print("TASK 2 PREPROCESSING: PASS")

train: null feature vectors=0; null labels=0
test: null feature vectors=0; null labels=0
Saved pipeline: /content/drive/MyDrive/wesad_task2_outputs/preprocessing_pipeline
Saved train data: /content/drive/MyDrive/wesad_task2_outputs/train_processed
Saved test data: /content/drive/MyDrive/wesad_task2_outputs/test_processed
Saved manifest: /content/drive/MyDrive/wesad_task2_outputs/task2_manifest.json
TASK 2 PREPROCESSING: PASS


### Capture the final partition, feature, label, and saved-output evidence


In [ ]:
print("Final Task 2 evidence")
print("Source rows loaded:", f"{raw_rows:,}")
print("Rows after filtering:", f"{rows_after_filter:,}")
print("Train rows:", f"{train_rows:,}")
print("Test rows:", f"{test_rows:,}")
print("Train/test participant overlap:", train_raw.select("subject_id").distinct().intersect(
    test_raw.select("subject_id").distinct()
).count())
print("Feature count:", len(FEATURES))
print("Pipeline stage order:", [type(stage).__name__ for stage in pipeline_model.stages])
print("Output directory:", RESULTS)
print("Output exists:", RESULTS.exists())
print("Output files:")
for p in sorted(RESULTS.iterdir()):
    print(p.name, "(directory)" if p.is_dir() else f"{p.stat().st_size:,} bytes")

Final Task 2 evidence
Source rows loaded: 60,807,600
Rows after filtering: 31,470,603
Train rows: 25,137,704
Test rows: 6,332,899
Train/test participant overlap: 0
Feature count: 8
Pipeline stage order: ['ImputerModel', 'VectorAssembler', 'StandardScalerModel', 'StringIndexerModel']
Output directory: /content/drive/MyDrive/wesad_task2_outputs
Output exists: True
Output files:
preprocessing_pipeline (directory)
task2_manifest.json 985 bytes
test_processed (directory)
train_processed (directory)
